In [1]:
import os

from dataset import *
from gpr import *
from train import *
from ace_extractor import *
from calculator import *
from plot import *
from dataset_analysis import *

import torch
from ase.io import read

# Dataset

In [2]:
dirlist = sorted(
    s for s in os.listdir('../seminar/Random_9layers/')
    if s.startswith('alloy')
)
slab_atoms = [read(f'../seminar/Random_9layers/{i}/OUTCAR') for i in dirlist]
slab_energy = [slab_atoms[i].get_potential_energy() for i in range(len(slab_atoms))]

# Configuration, Feature Extractor and Model Initiation

In [3]:
config = ACEConfig(
    elements=("Pd", "Pt"),
    max_order = 2,
    mindist=calc_mindist(slab_atoms[0]),
    shells=(0, 1.2, np.sqrt(2.6), np.sqrt(3.6), 2.1),
)

In [4]:
extractor = ClusterExpansion(config)

In [5]:
dataset = ACEDataset(
    atoms = slab_atoms,
    config = config,
    extractor = extractor,
    atom_indices = None, 
    target_y = slab_energy,
)

100%|██████████| 250/250 [00:07<00:00, 34.78it/s]


In [6]:
train_dataset, valid_dataset = train_valid_split(dataset,
                                                 train_fraction=0.8,
                                                 seed=45022
                                                 )

train_x, train_y = get_tensors_from_subset(train_dataset)
valid_x, valid_y = get_tensors_from_subset(valid_dataset)

In [7]:
model = SparseAtomicGPR(x_train=train_x,
                        M=450,
                        div=0.95,
                        init_lengthscale=25.0,
                        init_sigma2=1e-4,
                        init_outputscale = 3.0,
                        config=config,
                      )

100%|██████████| 14399/14399 [00:02<00:00, 5736.36it/s]

Selected 78 inducing points out of 450


# Training

In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-1)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=20,
)

In [9]:
history, best_rmse = train_sparse_atomic_gpr(
    train_x=train_x,
    train_y=train_y,
    valid_x=valid_x,
    valid_y=valid_y,
    optimizer=optimizer,
    scheduler=scheduler,
    model=model,
    n_epochs=201,
    model_path="sparse_atomic_gpr_slab.pt",
    device=torch.device("cpu"),
)

Iter 1/201 RMSE train: 0.017811 RMSE valid: 0.036293 MAE train: 0.014080 MAE valid: 0.027660 best RMSE valid: 0.036293 sigma2: 9.078e-05 ls_mean: 2.624e+01 outputscale: 3.305e+00 lr: 1.000e-01
Iter 51/201 RMSE train: 0.015484 RMSE valid: 0.034850 MAE train: 0.012663 MAE valid: 0.025569 best RMSE valid: 0.029309 sigma2: 2.596e-05 ls_mean: 3.086e+01 outputscale: 1.125e+01 lr: 5.000e-02
Iter 101/201 RMSE train: 0.014586 RMSE valid: 0.033644 MAE train: 0.011912 MAE valid: 0.024581 best RMSE valid: 0.029309 sigma2: 2.590e-05 ls_mean: 3.205e+01 outputscale: 1.128e+01 lr: 6.250e-03
Iter 151/201 RMSE train: 0.014416 RMSE valid: 0.033515 MAE train: 0.011771 MAE valid: 0.024471 best RMSE valid: 0.029309 sigma2: 2.590e-05 ls_mean: 3.230e+01 outputscale: 1.128e+01 lr: 1.563e-03
Iter 201/201 RMSE train: 0.014379 RMSE valid: 0.033440 MAE train: 0.011741 MAE valid: 0.024395 best RMSE valid: 0.029309 sigma2: 2.590e-05 ls_mean: 3.236e+01 outputscale: 1.128e+01 lr: 1.953e-04


# Results

In [10]:
model = SparseAtomicGPR(model_path = 'sparse_atomic_gpr_slab.pt')

In [11]:
fig, metrics = plot_results(
    model,
    train_x,
    train_y,
    valid_x,
    valid_y,
)

In [12]:
fig, X_emb, labels = plot_descriptor_space(
    model,
    train_x,
    valid_x)

# Calculator

In [13]:
calc = ACEGPRCalculator(model = model,
                        extractor = extractor)

In [14]:
atoms = atoms_near_carbon(slab_atoms[0])
atoms.calc = calc
atoms.get_potential_energy()

-155.901371684199

In [15]:
slab_energy[0]

-155.90072688

In [16]:
from ase.build import make_supercell

supercell = make_supercell(atoms, [[2,0,0],[0,2,0],[0,0,1]])
supercell.calc = calc
supercell.get_potential_energy()

-623.6054867367524

In [18]:
slab_energy[0]*4

-623.60290752

In [17]:
from ase.visualize import view
view(supercell, 'x3d')

<Popen: returncode: None args: ['/opt/anaconda3/envs/ace_gpr/bin/python', '-...>

In [22]:
read('structure_mc.xyz')[142]

Atom('Pd', [np.float64(5.97597), np.float64(7.96796), np.float64(36.91189)], index=142)